# Personal Wellbeing Time-Series Monitoring App

[Open this notebook in Google Colab](https://colab.research.google.com/github/Deolinda-Bogore/personal-wellbeing-time-series-monitor/blob/main/notebooks/personal_wellbeing_studentlife_pipeline.ipynb)

## StudentLife processed-data notebook

I use this notebook to inspect the data side of the wellbeing-monitoring project. Instead of asking for the full raw StudentLife `dataset.zip`, this version loads the processed CSV that is already stored in the repository.

That makes it easier to run in Colab: open the notebook, run the cells, and the data should load directly from GitHub.


## 1. What the data represents

The processed table is built from the Dartmouth StudentLife dataset. Each row represents one student on one day.

The project groups the available signals into five wellbeing domains:

| Domain | Features used in this project |
| --- | --- |
| Physical | sleep, activity, energy |
| Mental | stress, mood, energy |
| Social | social connection proxy |
| Occupational / academic | work or academic load, stress, energy |
| Digital | screen-time proxy, sleep, stress |

A few features are proxies, not perfect measurements. For example, screen time is only an approximate digital-load signal, and social connection is simplified into a daily score. I keep those assumptions visible because this project depends on turning imperfect daily signals into useful wellbeing trends.


## 2. Setup

This cell tries two paths:

1. Local repo path: `data/studentlife_wellbeing.csv`
2. GitHub raw CSV URL, which works in Colab

So you should not need to upload `dataset.zip` for this notebook.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

LOCAL_DATA_PATHS = [
    Path("../data/studentlife_wellbeing.csv"),
    Path("data/studentlife_wellbeing.csv"),
    Path("studentlife_wellbeing.csv"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/Deolinda-Bogore/personal-wellbeing-time-series-monitor/main/data/studentlife_wellbeing.csv"

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False


def load_processed_csv():
    for path in LOCAL_DATA_PATHS:
        if path.exists():
            print(f"Loaded local data from {path}")
            return pd.read_csv(path)

    try:
        print("Trying to load data from GitHub...")
        data = pd.read_csv(GITHUB_DATA_URL)
        print("Loaded data from GitHub")
        return data
    except Exception as error:
        print(f"GitHub load failed: {error}")

    if IN_COLAB:
        print("Upload the smaller processed CSV file: studentlife_wellbeing.csv")
        uploaded = files.upload()
        if uploaded:
            filename = next(iter(uploaded.keys()))
            print(f"Loaded uploaded file: {filename}")
            return pd.read_csv(filename)

    raise FileNotFoundError(
        "Could not load studentlife_wellbeing.csv. Use the GitHub notebook link, "
        "or upload data/studentlife_wellbeing.csv from the repository."
    )


df = load_processed_csv()
print(df.shape)
df.head()


## 3. Basic cleaning and schema check

Before scoring anything, I check that the expected columns are present and convert dates/numeric fields into the right types.


In [ ]:
REQUIRED_COLUMNS = [
    "student_id", "date", "sleep", "stress", "mood", "energy",
    "screenTime", "workHours", "activity", "social",
]

missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

clean = df[REQUIRED_COLUMNS].copy()
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
for column in REQUIRED_COLUMNS[2:]:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

clean = clean.dropna(subset=["student_id", "date"] + REQUIRED_COLUMNS[2:])
clean = clean.sort_values(["student_id", "date"]).reset_index(drop=True)

print(f"Rows: {len(clean):,}")
print(f"Students: {clean['student_id'].nunique():,}")
print(f"Date range: {clean['date'].min().date()} to {clean['date'].max().date()}")
clean.head()


## 4. Domain scoring

I convert raw daily signals into five domain scores from 0 to 100. These scores are simple and explainable on purpose, so the assumptions are easy to inspect.


In [ ]:
def clamp(series, lower=0, upper=100):
    return series.clip(lower=lower, upper=upper)


def add_domain_scores(data):
    scored = data.copy()
    sleep_score = clamp((scored["sleep"] / 8) * 100)
    stress_score = clamp(100 - (scored["stress"] - 1) * 11.1)
    mood_score = clamp(scored["mood"] * 10)
    energy_score = clamp(scored["energy"] * 10)
    social_score = clamp(scored["social"] * 10)
    activity_score = clamp((scored["activity"] / 45) * 100)
    screen_score = 100 - clamp((scored["screenTime"] - 5).clip(lower=0) * 10, 0, 60)
    work_score = 100 - clamp((scored["workHours"] - 8).clip(lower=0) * 12, 0, 60)

    scored["physical_score"] = sleep_score * 0.45 + activity_score * 0.30 + energy_score * 0.25
    scored["mental_score"] = stress_score * 0.50 + mood_score * 0.35 + energy_score * 0.15
    scored["social_score"] = social_score
    scored["occupational_score"] = work_score * 0.55 + stress_score * 0.30 + energy_score * 0.15
    scored["digital_score"] = screen_score * 0.70 + sleep_score * 0.15 + stress_score * 0.15
    scored["wellbeing_score"] = (
        scored["physical_score"] * 0.24
        + scored["mental_score"] * 0.28
        + scored["social_score"] * 0.14
        + scored["occupational_score"] * 0.17
        + scored["digital_score"] * 0.17
    )
    return scored.round(2)

scored = add_domain_scores(clean)
scored[["student_id", "date", "physical_score", "mental_score", "social_score", "occupational_score", "digital_score", "wellbeing_score"]].head()


## 5. Rule-based risk check

This is not a diagnosis. It is a transparent rule for finding days where the signals look concerning.

For the prototype, I mark a day as at risk when the overall wellbeing score is below 70.


In [ ]:
scored["risk_label"] = scored["wellbeing_score"] < 70

risk_summary = scored.groupby("student_id").agg(
    days=("date", "count"),
    mean_wellbeing=("wellbeing_score", "mean"),
    risk_days=("risk_label", "sum"),
).reset_index()
risk_summary["risk_rate"] = risk_summary["risk_days"] / risk_summary["days"]

risk_summary.sort_values("risk_rate", ascending=False).head(10)


## 6. Time-series features

For modeling, I add lag features and rolling averages. This is closer to the real problem because wellbeing today is often influenced by the previous few days.


In [ ]:
DOMAIN_COLUMNS = [
    "physical_score", "mental_score", "social_score",
    "occupational_score", "digital_score", "wellbeing_score",
]
RAW_FEATURES = ["sleep", "stress", "mood", "energy", "screenTime", "workHours", "activity", "social"]

model_df = scored.copy().sort_values(["student_id", "date"]).reset_index(drop=True)
model_df["day_of_week"] = model_df["date"].dt.dayofweek
model_df["days_since_student_start"] = (
    model_df["date"] - model_df.groupby("student_id")["date"].transform("min")
).dt.days

for column in RAW_FEATURES + DOMAIN_COLUMNS:
    grouped = model_df.groupby("student_id")[column]
    model_df[f"{column}_lag_1"] = grouped.shift(1)
    model_df[f"{column}_rolling_3"] = grouped.transform(lambda values: values.rolling(3, min_periods=2).mean())
    model_df[f"{column}_rolling_7"] = grouped.transform(lambda values: values.rolling(7, min_periods=2).mean())

model_df["next_day_wellbeing_score"] = model_df.groupby("student_id")["wellbeing_score"].shift(-1)
model_df["next_day_risk"] = model_df["next_day_wellbeing_score"] < 70
model_df = model_df.dropna(subset=["next_day_wellbeing_score"]).reset_index(drop=True)

print(model_df.shape)
model_df[["student_id", "date", "wellbeing_score", "next_day_wellbeing_score", "next_day_risk"]].head()


## 7. Visual checks

I use one student at a time to keep the chart readable.


In [ ]:
student_id = "u01"
student_plot = scored[scored["student_id"] == student_id].set_index("date")
plot_cols = ["physical_score", "mental_score", "social_score", "occupational_score", "digital_score", "wellbeing_score"]

fig, ax = plt.subplots(figsize=(14, 6))
student_plot[plot_cols].plot(ax=ax, linewidth=2)
ax.axhline(70, color="red", linestyle="--", alpha=0.6, label="risk threshold")
ax.set_title(f"Daily wellbeing domain scores for {student_id}")
ax.set_ylabel("Score, 0 to 100")
ax.set_xlabel("Date")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = scored[plot_cols].corr()
sns.heatmap(corr, annot=True, cmap="BrBG", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation between domain scores")
plt.tight_layout()
plt.show()


## 8. A simple next-day prediction model

This model predicts whether the next recorded day will be below the wellbeing threshold. I use a group split by student so the test set contains students the model did not train on.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

feature_columns = [
    column for column in model_df.columns
    if column not in ["student_id", "date", "risk_label", "next_day_risk", "next_day_wellbeing_score"]
]

x = model_df[feature_columns]
y = model_df["next_day_risk"].astype(int)
groups = model_df["student_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(x, y, groups=groups))

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

model.fit(x.iloc[train_idx], y.iloc[train_idx])
pred = model.predict(x.iloc[test_idx])
proba = model.predict_proba(x.iloc[test_idx])[:, 1]

metrics = {
    "accuracy": round(accuracy_score(y.iloc[test_idx], pred), 3),
    "precision": round(precision_score(y.iloc[test_idx], pred, zero_division=0), 3),
    "recall": round(recall_score(y.iloc[test_idx], pred, zero_division=0), 3),
    "f1": round(f1_score(y.iloc[test_idx], pred, zero_division=0), 3),
    "roc_auc": round(roc_auc_score(y.iloc[test_idx], proba), 3),
    "train_students": model_df.iloc[train_idx]["student_id"].nunique(),
    "test_students": model_df.iloc[test_idx]["student_id"].nunique(),
}
metrics


In [ ]:
importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.named_steps["classifier"].feature_importances_,
}).sort_values("importance", ascending=False)

importance.head(15)


## 9. Export

This final export is useful if I want to inspect the scored daily table outside the notebook.


In [ ]:
output_path = Path("processed_wellbeing_scored.csv")
scored.to_csv(output_path, index=False)
print(f"Saved scored data to {output_path}")
scored.head()


## What I learned from this pass

The useful part of the project is the pipeline: daily alignment, domain scoring, explainable labels, lag features, and evaluation across students.

Current limitations:

- The wellbeing score is rule-based, so the weights should be validated or learned later.
- Some signals are proxies, especially social and digital behavior.
- The model predicts a prototype risk label, not a clinical outcome.

Next steps would be to compare model families, add better temporal validation, and test whether domain-specific models explain risk more clearly than one overall model.
